# XLM-R — Vietnamese PII NER

## 1. Setup & Dependencies


In [ ]:
import os
import sys

REPO_DIR = "/content/Vietnamese-PII-NER" if os.path.exists("/content") else os.path.abspath("Vietnamese-PII-NER")
SETUP_MARKER = ".colab_deps_installed"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/PhucTang2005/Vietnamese-PII-NER.git "$REPO_DIR"

%cd $REPO_DIR

# Install dependencies once. Colab must restart after numpy/scikit-learn changes to avoid binary ABI issues.
if not os.path.exists(SETUP_MARKER):
    !pip install -r requirements.txt -q
    with open(SETUP_MARKER, "w", encoding="utf-8") as setup_file:
        setup_file.write("done")
    if os.environ.get("COLAB_RELEASE_TAG"):
        print("Dependencies installed. Restarting Colab runtime; run this cell again after restart.")
        os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed; skipping pip install.")


In [ ]:
# Import libraries and configure the runtime device.
import json
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

HF_XLMR_REPO = "Phuc2005/pii-xlm-r-base-ner"
TOKENIZED_DATA_DIR = "./data/tokenized_xlmr/"
CHECKPOINT_DIR = "./best_model_xlmr/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 2. Load Dataset


In [ ]:
# Load the Vietnamese PII dataset from HuggingFace Hub.
dataset = load_dataset("quynong/cs419-data")
print(dataset)
print("\nTrain sample:")
print(dataset['train'][0])


In [ ]:
# Build the BIO label schema from entity labels present in the dataset.
all_labels = set()
for split in dataset:
    for sample in dataset[split]:
        for ent in sample['privacy_mask']:
            all_labels.add(ent['label'])

all_labels = sorted(all_labels)
print(f"Found {len(all_labels)} entity types:")
print(all_labels)

label_list = ["O"]
for lbl in all_labels:
    label_list.append(f"B-{lbl}")
    label_list.append(f"I-{lbl}")

label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for idx, label in enumerate(label_list)}
num_labels = len(label_list)

print(f"\nTotal BIO labels: {num_labels}")
print(f"First 10: {label_list[:10]}")


## 3. Tokenization & Label Alignment


In [ ]:
# Configure the XLM-R tokenizer. XLM-R supports offset mapping on raw text.
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {MODEL_NAME} loaded.")


In [ ]:
# Tokenize raw text and align character-level entity spans to token-level BIO labels.
def tokenize_and_align_xlmr(examples):
    """Tokenize a batch of examples and align BIO labels using tokenizer offsets."""
    encoding = tokenizer(
        examples['source_text'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
        return_offsets_mapping=True,
    )

    all_token_labels = []
    for i, privacy_mask in enumerate(examples['privacy_mask']):
        source_text = examples['source_text'][i]
        offset_mapping = encoding['offset_mapping'][i]

        # Create character-level BIO labels from privacy_mask spans.
        char_labels = ['O'] * len(source_text)
        for ent in privacy_mask:
            start, end, label = ent['start'], ent['end'], ent['label']
            if start < len(source_text) and end <= len(source_text):
                char_labels[start] = f"B-{label}"
                for char_idx in range(start + 1, end):
                    char_labels[char_idx] = f"I-{label}"

        # Convert character-level labels to token-level labels.
        token_labels = []
        for tok_start, tok_end in offset_mapping:
            if tok_start == 0 and tok_end == 0:
                token_labels.append(-100)
            else:
                label = char_labels[tok_start] if tok_start < len(char_labels) else 'O'
                token_labels.append(label2id.get(label, label2id['O']))

        all_token_labels.append(token_labels)

    encoding['labels'] = all_token_labels
    encoding.pop('offset_mapping')
    return encoding


In [ ]:
# Tokenize all dataset splits and save the processed dataset inside the repo.
tokenized_dataset = dataset.map(
    tokenize_and_align_xlmr,
    batched=True,
    batch_size=32,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing XLM-R",
)

tokenized_dataset.save_to_disk(TOKENIZED_DATA_DIR)

print(tokenized_dataset)
print(f"\nSample input_ids length: {len(tokenized_dataset['train'][0]['input_ids'])}")
print(f"Sample labels length: {len(tokenized_dataset['train'][0]['labels'])}")
print(f"Tokenized dataset saved to: {TOKENIZED_DATA_DIR}")


## 4. Model Definition


In [ ]:
# Load XLM-R for token classification with the dataset-specific BIO label schema.
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
print(f"Model loaded: {MODEL_NAME} with {num_labels} labels")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 5. Training


In [ ]:
# Configure HuggingFace Trainer for XLM-R fine-tuning.
training_args = TrainingArguments(
    output_dir="./xlmr-ner-pii",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("Training arguments configured for XLM-R.")


In [ ]:
# Compute entity-level NER metrics and sentence-level entity-detection metrics.
def compute_metrics(eval_preds):
    """Compute NER micro metrics and binary sentence-level PII detection metrics."""
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Build seqeval inputs by ignoring padding and special-token labels.
    true_labels = []
    true_predictions = []

    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags = []
        true_tags = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            pred_tags.append(id2label[pred_id])
            true_tags.append(id2label[label_id])
        true_predictions.append(pred_tags)
        true_labels.append(true_tags)

    ner_precision = precision_score(true_labels, true_predictions, average='micro')
    ner_recall = recall_score(true_labels, true_predictions, average='micro')
    ner_f1 = f1_score(true_labels, true_predictions, average='micro')

    # Convert token-level labels to a binary sentence-level PII signal.
    cls_true = []
    cls_pred = []

    for pred_seq, label_seq in zip(predictions, labels):
        has_entity_true = any(
            label_id not in (-100, label2id['O']) for label_id in label_seq
        )
        has_entity_pred = any(
            pred_id != label2id['O']
            for pred_id, label_id in zip(pred_seq, label_seq)
            if label_id != -100
        )
        cls_true.append(int(has_entity_true))
        cls_pred.append(int(has_entity_pred))

    cls_precision, cls_recall, cls_f1, _ = precision_recall_fscore_support(
        cls_true, cls_pred, average='binary', zero_division=0
    )
    cls_accuracy = accuracy_score(cls_true, cls_pred)

    return {
        "precision": ner_precision,
        "recall": ner_recall,
        "f1": ner_f1,
        "cls_accuracy": cls_accuracy,
        "cls_precision": cls_precision,
        "cls_recall": cls_recall,
        "cls_f1": cls_f1,
    }

print("Metrics function loaded.")


In [ ]:
# Train the model on the tokenized training split and validate each epoch.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Trainer ready. Starting training XLM-R...")
trainer.train()


## 6. Evaluation


In [ ]:
# Run final evaluation on the validation split.
eval_results = trainer.evaluate()

print("=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)
print("\n" + "-" * 40)
print("NER Entity-Level (Micro):")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")
print(f"  F1:        {eval_results['eval_f1']:.4f}")
print("\n" + "-" * 40)
print("Classification (Has Entity):")
print(f"  Accuracy:  {eval_results['eval_cls_accuracy']:.4f}")
print(f"  Precision: {eval_results['eval_cls_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_cls_recall']:.4f}")
print(f"  F1:        {eval_results['eval_cls_f1']:.4f}")
print("=" * 60)


In [ ]:
# Generate a detailed seqeval report per entity type on the validation split.
eval_output = trainer.predict(tokenized_dataset['validation'])
predictions = np.argmax(eval_output.predictions, axis=-1)
labels = eval_output.label_ids

true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    pred_tags = []
    true_tags = []
    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id == -100:
            continue
        pred_tags.append(id2label[pred_id])
        true_tags.append(id2label[label_id])
    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

report = classification_report(true_labels, true_predictions, digits=4)
print("Detailed NER Classification Report for XLM-R (per entity type):")
print(report)


## 7. Inference


In [ ]:
# Load model and run inference on sample text.
inference_tokenizer = AutoTokenizer.from_pretrained(HF_XLMR_REPO)
inference_model = AutoModelForTokenClassification.from_pretrained(HF_XLMR_REPO)
inference_model.to(device)
inference_model.eval()

inference_id2label = inference_model.config.id2label


def predict_entities(text):
    """Run XLM-R NER inference on a single Vietnamese text."""
    encoding = inference_tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors='pt',
        return_offsets_mapping=True,
    )

    # Keep offsets on CPU for span extraction and move model inputs to the runtime device.
    offset_mapping = encoding.pop('offset_mapping')[0].tolist()
    inputs = {key: value.to(device) for key, value in encoding.items()}

    with torch.no_grad():
        outputs = inference_model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()

    # Merge BIO token predictions into entity spans in the original text.
    entities = []
    current_entity = None

    for pred_id, (start, end) in zip(predictions, offset_mapping):
        if start == 0 and end == 0:
            continue
        label = inference_id2label[pred_id]

        if label.startswith('B-'):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'label': label[2:],
                'start': start,
                'end': end,
                'text': text[start:end],
            }
        elif label.startswith('I-') and current_entity:
            current_entity['end'] = end
            current_entity['text'] = text[current_entity['start']:end]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    return entities


text = "Xin chào, tôi là Nguyễn Văn An, SĐT 0912345678, email example@gmail.com"
entities = predict_entities(text)

print(f"Text: {text}")
print(f"\nEntities found: {len(entities)}")
for ent in entities:
    print(f"  [{ent['label']}] \"{ent['text']}\" (pos {ent['start']}-{ent['end']})")


## 8. Save & Export


In [ ]:
# Save the fine-tuned model and tokenizer to the repo checkpoint directory.
trainer.save_model(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Model saved to: {CHECKPOINT_DIR}")
